# SPMV CSR

In [2]:
import time
import numpy as np
import scipy.sparse as sp
from pynq import Overlay, allocate

# =====================================================================
# 1. Initialization and IP Setup
# =====================================================================

# Load the overlay (Update the path to your actual .bit file)
bitstream_path = '/home/xilinx/spmv_csr_optimized/spmv_csr_optimized.bit'
print(f"Loading overlay from {bitstream_path}...")
overlay = Overlay(bitstream_path)
print('Overlay loaded!')

# Get handler of the spmv IP
spmv_ip = overlay.spmv_csr_0 


# =====================================================================
# 2. Problem Setup & Matrix Generation
# =====================================================================
print("\nRunning SPMV CSR testbench...")

num_rows = 1024
num_cols = 1024
density  = 0.05 # 5% non-zeros
PACK_SIZE = 16  # Hardware fetches 16 elements (512-bit) at a time

print(f"Generating random {num_rows}x{num_cols} sparse matrix (Density: {density})...")

np.random.seed(42)

# Generate Random CSR Matrix
sparse_matrix = sp.random(num_rows, num_cols, density=density, format='csr', dtype=np.float32, random_state=42)

# Extract CSR Arrays
A_values_data    = sparse_matrix.data
A_col_index_data = sparse_matrix.indices.astype(np.int32)
A_row_index_data = sparse_matrix.indptr.astype(np.int32)

# Input vector 'x'
x_data = np.random.rand(num_cols).astype(np.float32)

nnz = sparse_matrix.nnz
print(f"Generated {nnz} Non-Zero (NNZ) elements.")

# =====================================================================
# 3. Buffer Allocation and Padding
# =====================================================================
# The hardware reads/writes in chunks of 16. 
# We must pad buffers to avoid out-of-bounds AXI reads.
padded_nnz = ((nnz + PACK_SIZE - 1) // PACK_SIZE) * PACK_SIZE
padded_cols = ((num_cols + PACK_SIZE - 1) // PACK_SIZE) * PACK_SIZE
padded_rows = ((num_rows + PACK_SIZE - 1) // PACK_SIZE) * PACK_SIZE
padded_row_index = ((len(A_row_index_data) + PACK_SIZE - 1) // PACK_SIZE) * PACK_SIZE

print(f"Allocating Contiguous Memory...")
A_val_buf = allocate(shape=(padded_nnz,), dtype=np.float32)
A_col_buf = allocate(shape=(padded_nnz,), dtype=np.int32)
A_row_buf = allocate(shape=(padded_row_index,), dtype=np.int32)
x_buf     = allocate(shape=(padded_cols,), dtype=np.float32)
y_buf     = allocate(shape=(num_rows,), dtype=np.float32)

# Initialize Buffers with generated data
A_val_buf[:nnz] = A_values_data
A_col_buf[:nnz] = A_col_index_data
A_row_buf[:len(A_row_index_data)] = A_row_index_data
x_buf[:num_cols] = x_data

# Zero-out the padding lanes
A_val_buf[nnz:] = 0.0
A_col_buf[nnz:] = 0
A_row_buf[len(A_row_index_data):] = 0
x_buf[num_cols:] = 0.0

# Flush CPU cache to DDR
A_val_buf.sync_to_device()
A_col_buf.sync_to_device()
A_row_buf.sync_to_device()
x_buf.sync_to_device()
y_buf.sync_to_device()

# =====================================================================
# 4. Hardware Execution
# =====================================================================
# Register Offsets based on provided header
ADDR_AP_CTRL    = 0x00
ADDR_NUM_ROWS   = 0x10
ADDR_NUM_COLS   = 0x18
ADDR_NNZ        = 0x20
ADDR_A_ROW_IDX  = 0x28
ADDR_A_COL_IDX  = 0x34
ADDR_A_VALUES   = 0x40
ADDR_X          = 0x4c
ADDR_Y          = 0x58

def write_64bit_address(ip, base_offset, address):
    """Writes a 64-bit physical address across two 32-bit registers."""
    ip.write(base_offset, address & 0xFFFFFFFF)
    ip.write(base_offset + 0x04, (address >> 32) & 0xFFFFFFFF)

# Write Scalar Parameters
print("\nConfiguring Registers...")
spmv_ip.write(ADDR_NUM_ROWS, num_rows)
spmv_ip.write(ADDR_NUM_COLS, num_cols)
spmv_ip.write(ADDR_NNZ, nnz)

# Write 64-bit Memory Pointers
write_64bit_address(spmv_ip, ADDR_A_ROW_IDX, A_row_buf.device_address)
write_64bit_address(spmv_ip, ADDR_A_COL_IDX, A_col_buf.device_address)
write_64bit_address(spmv_ip, ADDR_A_VALUES, A_val_buf.device_address)
write_64bit_address(spmv_ip, ADDR_X, x_buf.device_address)
write_64bit_address(spmv_ip, ADDR_Y, y_buf.device_address)

print("Starting Hardware Accelerator...")
start_time = time.time()

# Set ap_start
spmv_ip.write(ADDR_AP_CTRL, 0x01)

# Poll ap_done (bit 1)
while (spmv_ip.read(ADDR_AP_CTRL) & 0x02) == 0:
    pass

end_time = time.time()
hw_duration = (end_time - start_time) * 1000
print(f"HW Execution Time: {hw_duration:.4f} ms")

# Retrieve Results
y_buf.sync_from_device()
y_hw = np.array(y_buf)

# ==========================================
# 5. Software Reference & Verification
# ==========================================
print("\nRunning Software Reference...")
      
sw_start = time.time()

# Using scipy's optimized dot product for a fair/fast comparison on 256x256
y_sw = sparse_matrix.dot(x_data) 

sw_end = time.time()
sw_duration = (sw_end - sw_start) * 1000
print(f"SW Execution Time: {sw_duration:.4f} ms")

# Verification
print("\n--- Results ---")

print("\nResult Vector (y) [Showing first 10 rows]:")
print("Row | SW Ref       | HW Accel")
print("----------------------------------")
for i in range(min(10, num_rows)):
    print(f"{i:3} | {y_sw[i]:10.4f} | {y_hw[i]:10.4f}")
print("...\n")

# Use numpy's allclose to check with a relative tolerance (matches your 0.05% margin logic)
# rtol=0.0005 is equivalent to your C++ logic: diff / expected > 0.0005f
is_match = np.allclose(y_hw, y_sw, rtol=0.0005, atol=1e-5)

if is_match:
    print(f">>> SUCCESS: All {num_rows} hardware results match software reference! <<<")
else:
    # Print out mismatch details
    diffs = np.abs(y_hw - y_sw)
    rel_errors = diffs / (np.abs(y_sw) + 1e-8)
    mismatches = np.where(rel_errors > 0.0005)[0]
    
    print(f">>> ERROR: {len(mismatches)} Mismatches detected! <<<")
    for i in mismatches[:10]:
        print(f"Mismatch at row {i}: HW = {y_hw[i]:.6f}, SW = {y_sw[i]:.6f} (Diff = {diffs[i]:.6f})")
    
# Free contiguous memory buffers
A_val_buf.close()
A_col_buf.close()
A_row_buf.close()
x_buf.close()
y_buf.close()

Loading overlay from /home/xilinx/spmv_csr_optimized/spmv_csr_optimized.bit...
Overlay loaded!

Running SPMV CSR testbench...
Generating random 1024x1024 sparse matrix (Density: 0.05)...
Generated 52429 Non-Zero (NNZ) elements.
Allocating Contiguous Memory...

Configuring Registers...
Starting Hardware Accelerator...
HW Execution Time: 2.0692 ms

Running Software Reference...
SW Execution Time: 1.0157 ms

--- Results ---

Result Vector (y) [Showing first 10 rows]:
Row | SW Ref       | HW Accel
----------------------------------
  0 |    14.3740 |    14.3740
  1 |     9.8293 |     9.8293
  2 |    13.0148 |    13.0148
  3 |    18.2517 |    18.2517
  4 |    16.5824 |    16.5824
  5 |    12.3488 |    12.3488
  6 |    11.6145 |    11.6145
  7 |    12.4067 |    12.4067
  8 |    10.1840 |    10.1840
  9 |    10.9560 |    10.9560
...

>>> SUCCESS: All 1024 hardware results match software reference! <<<
